## Mô Hình M2: VGG-ResNet Sâu Từ Số 0 (BN, Residual, Spatial Dropout, GAP)

Báo cáo kiến trúc, huấn luyện và đánh giá mô hình CNN sâu tự thiết kế **M2VggRes** trên tập dữ liệu biển báo giao thông GTSRB.

**Mục tiêu chính**:
1. Khắc phục triệt để điểm nghẽn của mô hình M1 LeNet thông qua 5 lựa chọn thiết kế kiến trúc hiện đại.
2. Chứng minh hiệu quả của **Global Average Pooling (GAP)**: M2 sâu gấp hơn 4 lần (9 lớp tích chập so với 2 lớp của M1) nhưng tổng tham số chỉ bằng **51%** so với M1 (1.24M vs 2.42M tham số).
3. Tích hợp khối **ResidualBlock** với kết nối tắt (Skip Connection) giải quyết hiện tượng suy thoái (degradation) và làm mượt bề mặt hàm mất mát.

## 1. Khởi tạo Môi trường và Thiết lập Cấu hình Tham số

### 1.1. Khởi tạo thư viện và chuyển thư mục làm việc về gốc

In [1]:
import os
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_recall_fscore_support

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2

# Thiết lập thư mục gốc của dự án
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
print(f"Thư mục làm việc: {Path.cwd()}")

Thư mục làm việc: d:\code\Python\DSEB_ex\DL\gtsrb-model-comparison


### 1.2. Thiết lập cờ điều khiển và tham số thực thi
* `FORCE_RETRAIN = False`: Mặc định nạp checkpoint tối ưu đã huấn luyện tại `checkpoints/m2_vggres_best.pt`.
* `DEMO_EPOCHS = 2`: Số epoch chạy thử nếu cần kiểm tra quy trình huấn luyện.

In [2]:
FORCE_RETRAIN = False       # Đặt True nếu muốn huấn luyện lại từ đầu
DEMO_EPOCHS = 2            # Số epoch chạy demo
FULL_EPOCHS = 40           # Số epoch huấn luyện hoàn chỉnh
BATCH_SIZE = 64
LR = 0.001

INDEX_CSV = Path("data/processed/index.csv")
CKPT_PATH = Path("checkpoints/m2_vggres_best.pt")
IMG_SIZE = 48

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Thiết bị tính toán: {DEVICE}")

Thiết bị tính toán: cpu


## 2. Pipeline Dữ Liệu (Dataset & DataLoader)

### 2.1. Chuẩn hóa và Augmentation
Mô hình M2 được huấn luyện từ đầu (from scratch) trên GTSRB nên sử dụng thống kê phân phối của tập train GTSRB (`mean=[0.4597, 0.4254, 0.4376]`, `std=[0.2670, 0.2607, 0.2691]`).

In [3]:
NUM_CLASSES = 43
GTSRB_MEAN = (0.4597, 0.4254, 0.4376)
GTSRB_STD = (0.2670, 0.2607, 0.2691)

def _build_normalize(mean=GTSRB_MEAN, std=GTSRB_STD) -> v2.Transform:
    return v2.Compose([
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=list(mean), std=list(std)),
    ])

def build_train_transform(img_size: int = 48) -> v2.Transform:
    return v2.Compose([
        v2.RandomAffine(degrees=15.0, translate=(0.10, 0.10), scale=(0.9, 1.1),
                        interpolation=v2.InterpolationMode.BILINEAR, fill=0),
        v2.ColorJitter(brightness=0.20, contrast=0.20),
    ])

### 2.2. Lớp Dữ Liệu và Khởi tạo DataLoaders
Nạp trực tiếp từ file cache `.npy` ở độ phân giải $48 \times 48$.

In [4]:
class GTSRBDataset(Dataset):
    def __init__(self, index_csv: str | Path, split: str, img_size: int = 48,
                 preprocess: str = "clahe", processed_dir: str | Path = "data/processed",
                 cache_size: int = 48, transform=None) -> None:
        self.split = split
        self.img_size = img_size
        self.transform = transform

        frame = pd.read_csv(index_csv)
        self.frame = frame[frame["split"] == split].reset_index(drop=True)
        self._cache_rows = frame.index[frame["split"] == split].to_numpy()
        self.normalizer = _build_normalize(GTSRB_MEAN, GTSRB_STD)

        path = Path(processed_dir) / f"images_{cache_size}_{preprocess}.npy"
        self.cache = np.load(path, mmap_mode="r")

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, i: int):
        image = np.array(self.cache[self._cache_rows[i]], copy=True)
        label = int(self.frame.iloc[i]["class_id"])
        tensor = torch.from_numpy(np.ascontiguousarray(image)).permute(2, 0, 1)

        if self.transform is not None:
            tensor = self.transform(tensor)

        return self.normalizer(tensor), label

def make_loader(index_csv: str | Path, split: str, img_size: int = 48, batch_size: int = 64,
                transform=None, num_workers: int = 0) -> DataLoader:
    ds = GTSRBDataset(index_csv, split, img_size=img_size, transform=transform)
    is_train = split == "train"
    return DataLoader(ds, batch_size=batch_size, shuffle=is_train, drop_last=is_train, num_workers=num_workers)

train_tf = build_train_transform(IMG_SIZE)
train_loader = make_loader(INDEX_CSV, "train", img_size=IMG_SIZE, batch_size=BATCH_SIZE, transform=train_tf)
val_loader = make_loader(INDEX_CSV, "val", img_size=IMG_SIZE, batch_size=BATCH_SIZE)
test_loader = make_loader(INDEX_CSV, "test", img_size=IMG_SIZE, batch_size=BATCH_SIZE)

print(f"DataLoaders: Train={len(train_loader.dataset)}, Val={len(val_loader.dataset)}, Test={len(test_loader.dataset)} mẫu.")

DataLoaders: Train=31379, Val=7830, Test=12630 mẫu.


## 3. Năm Lựa Chọn Thiết Kế Kiến Trúc của Mô Hình M2

### 3.1. Phân tích chi tiết 5 cải tiến kỹ thuật
1. **Hai lớp Conv 3x3 thay thế một lớp Conv 5x5 (Triết lý VGG)**:
   * Hai lớp Conv 3x3 có cùng Receptive Field $5 \times 5$ ($RF = 2n + 1 = 2(2) + 1 = 5$), nhưng giảm tham số: $2 \times (3 \times 3 \times C^2) = 18C^2$ so với $5 \times 5 \times C^2 = 25C^2$ (giảm **28%**).
   * Tích hợp 2 hàm phi tuyến ReLU liên tiếp giúp không gian biểu diễn phong phú hơn.
2. **Batch Normalization (Conv $\rightarrow$ BN $\rightarrow$ ReLU)**:
   * Chuẩn hóa activation từng kênh, làm mượt bề mặt hàm mất mát (Santurkar et al., 2018), cho phép sử dụng learning rate lớn hơn và hội tụ ổn định hơn.
3. **Residual Connection ($y = F(x) + x$)**:
   * Gradient luôn có đường truyền tắt $\partial y / \partial x = \partial F / \partial x + 1$ (chống triệt tiêu gradient).
   * Khắc phục hiện tượng suy thoái (degradation) khi tăng độ sâu mạng.
4. **Spatial Dropout (`nn.Dropout2d`)**:
   * Dropout thông thường loại bỏ pixel độc lập, vốn kém hiệu quả trên feature map do các pixel lân cận có tương quan không gian rất cao.
   * `Dropout2d` loại bỏ **toàn bộ một kênh đặc trưng** ngẫu nhiên, buộc mô hình không phụ thuộc vào một detector đơn lẻ.
5. **Global Average Pooling (GAP) thay thế Flatten + Linear**:
   * Feature map $(B, 256, 3, 3) \rightarrow$ Vector $(B, 256)$ qua trung bình không gian.
   * Giảm số tham số của Classifier từ 2.359.552 xuống chỉ còn $256 \times 43 = 11.051$ tham số (giảm **~213 lần**).

### 3.2. Cài đặt Khối `ResidualBlock`

In [5]:
class ResidualBlock(nn.Module):
    """Khối phần dư gồm 2 lớp Conv 3x3 + BatchNorm và nhánh kết nối tắt (Skip Connection)."""
    def __init__(self, in_ch: int, out_ch: int) -> None:
        super().__init__()
        # bias=False vì đứng trước BatchNorm (BN có tham số beta đóng vai trò của bias)
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)

        if in_ch == out_ch:
            self.skip = nn.Identity()
        else:
            # Chiếu số kênh khi thay đổi kích thước kênh bằng Conv 1x1
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.skip(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)  # Phép cộng trước, ReLU sau (theo ResNet gốc)

### 3.3. Cài đặt Mô hình Tổng thể `M2VggRes`

In [6]:
class M2VggRes(nn.Module):
    """Kiến trúc VGG-ResNet 4 stage kết hợp Spatial Dropout và Global Average Pooling."""
    def __init__(self, num_classes: int = NUM_CLASSES, img_size: int = 48) -> None:
        super().__init__()
        # Stem layer
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )

        # 4 Stages: Mỗi stage gồm ResidualBlock + MaxPool + Spatial Dropout (p tăng dần)
        self.stages = nn.Sequential(
            nn.Sequential(
                ResidualBlock(32, 32),
                nn.MaxPool2d(2),          # 48x48 -> 24x24
                nn.Dropout2d(0.1),
            ),
            nn.Sequential(
                ResidualBlock(32, 64),
                nn.MaxPool2d(2),          # 24x24 -> 12x12
                nn.Dropout2d(0.2),
            ),
            nn.Sequential(
                ResidualBlock(64, 128),
                nn.MaxPool2d(2),          # 12x12 -> 6x6
                nn.Dropout2d(0.3),
            ),
            nn.Sequential(
                ResidualBlock(128, 256),
                nn.MaxPool2d(2),          # 6x6 -> 3x3
                nn.Dropout2d(0.3),
            ),
        )

        # Classification Head: GAP + Linear
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),   # (B, 256, 3, 3) -> (B, 256, 1, 1)
            nn.Flatten(),             # -> (B, 256)
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

        self._init_weights()

    def _init_weights(self) -> None:
        """Khởi tạo trọng số Kaiming He Normal phù hợp cho mạng sử dụng hàm kích hoạt ReLU."""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.stages(self.stem(x)))


### 3.4. So Sánh Định Lượng Tham Số: M2 vs M1

In [7]:
model_m2 = M2VggRes()
total_m2 = sum(p.numel() for p in model_m2.parameters())
total_m1 = 2424299  # Tham số M1 đo được ở notebook trước

print("=" * 60)
print(f"SO SÁNH THAM SỐ GIỮA M1 (LeNet) VÀ M2 (VGG-ResNet)")
print("=" * 60)
print(f"M1 LeNet (2 lớp Conv)        : {total_m1:,} tham số")
print(f"M2 VGG-ResNet (9 lớp Conv)    : {total_m2:,} tham số")
print(f"Tỷ lệ M2 / M1                 : {total_m2 / total_m1 * 100:.2f}%")
print(f"Tham số lớp FC cuối của M2    : {256 * 43 + 43:,} tham số (so với 2.359.552 của M1)")
print("=" * 60)

SO SÁNH THAM SỐ GIỮA M1 (LeNet) VÀ M2 (VGG-ResNet)
M1 LeNet (2 lớp Conv)        : 2,424,299 tham số
M2 VGG-ResNet (9 lớp Conv)    : 1,237,451 tham số
Tỷ lệ M2 / M1                 : 51.04%
Tham số lớp FC cuối của M2    : 11,051 tham số (so với 2.359.552 của M1)


## 4. Hàm Mất Mát và Các Hàm Huấn Luyện

Áp dụng Cross-Entropy Loss kết hợp Label Smoothing $\epsilon = 0.1$ nhằm giảm thiểu sự tự tin cực đoan (over-confidence) của mô hình.

In [8]:
@torch.no_grad()
def predict(model: nn.Module, loader: DataLoader, device: torch.device) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    probs, targets = [], []
    for images, y in loader:
        logits = model(images.to(device))
        probs.append(torch.softmax(logits.float(), dim=1).cpu())
        targets.append(y)
    return torch.cat(targets).numpy(), torch.cat(probs).numpy()

def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray) -> dict:
    y_pred = y_prob.argmax(axis=1)
    top1 = float((y_pred == y_true).mean())
    top5_idx = np.argsort(-y_prob, axis=1)[:, :5]
    top5 = float((top5_idx == y_true[:, None]).any(axis=1).mean())
    macro_f1 = float(f1_score(y_true, y_pred, labels=list(range(NUM_CLASSES)), average="macro", zero_division=0))
    return {"top1": top1, "top5": top5, "macro_f1": macro_f1, "y_true": y_true, "y_pred": y_pred}

def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> dict:
    y_true, y_prob = predict(model, loader, device)
    return compute_metrics(y_true, y_prob)

def train_one_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module,
                    optimizer: torch.optim.Optimizer, device: torch.device) -> float:
    model.train()
    total_loss = 0.0
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(targets)
    return total_loss / len(loader.dataset)

def train_m2(epochs: int = 40, lr: float = 1e-3, save_path: Path = CKPT_PATH) -> M2VggRes:
    model = M2VggRes().to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

    best_f1 = 0.0
    save_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Bắt đầu huấn luyện M2VggRes ({epochs} epochs)...\n")

    for epoch in range(1, epochs + 1):
        loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_metrics = evaluate(model, val_loader, DEVICE)
        scheduler.step()

        val_f1 = val_metrics["macro_f1"]
        val_top1 = val_metrics["top1"]
        is_best = val_f1 > best_f1
        if is_best:
            best_f1 = val_f1
            torch.save(model.state_dict(), save_path)

        tag = "* [BEST]" if is_best else ""
        print(f"Epoch {epoch:02d}/{epochs:02d} | Train Loss: {loss:.4f} | Val Top-1: {val_top1*100:.2f}% | Val Macro-F1: {val_f1:.4f} {tag}")

    print(f"\nHuấn luyện hoàn tất. Đã lưu checkpoint tốt nhất tại: {save_path}")
    ckpt = torch.load(save_path, map_location=DEVICE, weights_only=False)
    state = ckpt["model_state"] if isinstance(ckpt, dict) and "model_state" in ckpt else ckpt
    model.load_state_dict(state)
    return model

## 5. Thực Thi: Nạp Checkpoint Hoặc Huấn Luyện Mới

In [9]:
model = M2VggRes().to(DEVICE)

if not FORCE_RETRAIN and CKPT_PATH.exists():
    print(f"[Tài nguyên sẵn sàng] Nạp checkpoint mô hình từ {CKPT_PATH}")
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    state_dict = ckpt["model_state"] if isinstance(ckpt, dict) and "model_state" in ckpt else ckpt
    model.load_state_dict(state_dict)
    print("Nạp trọng số thành công.")
else:
    epochs_to_run = DEMO_EPOCHS if not FORCE_RETRAIN else FULL_EPOCHS
    print(f"[Thực thi] Bắt đầu huấn luyện mô hình ({epochs_to_run} epoch)...\n")
    model = train_m2(epochs=epochs_to_run, lr=LR)

[Tài nguyên sẵn sàng] Nạp checkpoint mô hình từ checkpoints\m2_vggres_best.pt
Nạp trọng số thành công.


## 6. Báo Cáo Kết Quả Đánh Giá Trên Tập Test Chính Thức

In [10]:
test_metrics = evaluate(model, test_loader, DEVICE)

print("=" * 50)
print("KẾT QUẢ KIỂM THỬ MÔ HÌNH M2 (VGG-ResNet)")
print("=" * 50)
print(f"Top-1 Accuracy : {test_metrics['top1'] * 100:.2f}%")
print(f"Top-5 Accuracy : {test_metrics['top5'] * 100:.2f}%")
print(f"Macro-F1       : {test_metrics['macro_f1']:.4f}")
print("=" * 50)

# So sánh nhanh với mốc M1 LeNet (98.46% Top-1, 0.9778 Macro-F1)
delta_top1 = (test_metrics['top1'] - 0.9846) * 100
delta_f1 = (test_metrics['macro_f1'] - 0.9778)
print(f"Độ cải thiện so với M1 LeNet:")
print(f"  Δ Top-1   : +{delta_top1:.2f}%")
print(f"  Δ Macro-F1: +{delta_f1:.4f}")

KẾT QUẢ KIỂM THỬ MÔ HÌNH M2 (VGG-ResNet)
Top-1 Accuracy : 99.26%
Top-5 Accuracy : 99.94%
Macro-F1       : 0.9880
Độ cải thiện so với M1 LeNet:
  Δ Top-1   : +0.80%
  Δ Macro-F1: +0.0102
